# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Jericho-Ram/FlyRank-Internship-ML/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

### Finding: ML Appendix — Feature Importance, "What Predicts Health?" (p.27)

**Claim:** Random Forest feature importance for predicting health score, ranked Average
Position (43%), Impressions (32%), Scroll Depth (15%), CTR (8%), everything else ~0%.

**Where the label comes from:** Health Score is a FlyRank composite defined earlier in the
paper as Impressions (30pts) + Position (30pts) + CTR (20pts) + Scroll Depth (20pts). Three
of the four top "predictors" — Position, Impressions, CTR — are literally the inputs to the
formula being predicted. This is exactly the label-derived-feature pattern from the leakage
taxonomy: the label was computed FROM these columns, and the columns are also in the
features.

**Does the validation design carry the claim?** The paper is honest about this up front —
"the target itself is partly constructed from some of these inputs, so importance is
descriptive rather than causal" — which is the right caveat and better disclosure than a lot
of published feature-importance charts get. But "holdout-tested" is still doing some
work in the framing that I don't think it can support here: holdout testing checks whether a
*mapping* generalizes to unseen rows, not whether the mapping is *informative*. If health
score is a close-to-linear function of position/impressions/CTR/scroll, a Random Forest will
recover that function almost perfectly on holdout data regardless of whether anything about
the real world is being learned — the same reason my own Test B (Week 3) sends AUC from
0.645 to 1.0 the moment I put `trend_pct` back into my features. High holdout accuracy on a
near-tautological mapping is not evidence the mapping means anything beyond the formula
itself.

**Constructive suggestion:** the paper's own leakage-taxonomy-style fix is one train run
away — refit the Random Forest using *only* the near-zero-importance features (Clicks,
Sessions, Content Age, Word Count, Days Visible, AI Sessions) and report that importance
ranking instead, or alongside. That's the "train once WITH the suspect, once WITHOUT" test,
and it would show whether there's a real, non-circular signal underneath the top layer,
rather than leaving the chart dominated by the score rediscovering its own definition.

### Finding: ML Appendix — Growth & Classification (p.29)

**Claim:** Logistic regression, "71% holdout accuracy," describing which features separate
growing from declining pages. Content Age, Days Since Update, and Days Visible are reported
as the strongest signals.

**Where the label comes from:** Trend Direction is defined earlier as computed from
30-day-vs-previous-30-day impression change (Up >10%, Down >10%, Stable within ±10%). That's
a portfolio-wide label spanning 57 brands.

**Does the validation design carry the claim?** Two open questions, both squarely inside
what this week's skill calls out: 1) *Split honesty* — the paper doesn't say whether the
71% holdout is a random row split or a split grouped by brand. With 57 brands and pages
that plausibly share brand-level characteristics (template, publishing cadence, niche), a
random split risks the exact client-memorization failure my own Week-6 audit just measured
on my own model — a 0.122 AUC gap between random and grouped splits on the same feature
set. Given that gap turned out to be large for me on a much smaller dataset (32 clients), I
wouldn't assume it's negligible here without being told the split method. 2) *Window
overlap* — "Days Visible" is reported as one of the strongest positive signals, but the
paper doesn't specify whether it's counted over a window that's disjoint from the 30-day
window used to compute the growth/decline label, or whether it overlaps it. If it overlaps,
that's the same "future/overlapping windows" pattern I excluded `clicks_last_30d` /
`sessions_last_30d` for in my own Week-5 strict feature set.

**Constructive suggestion:** state the split method (random vs. grouped-by-brand) next to
the 71% figure, and confirm — or adjust — the window definition for `days_visible` relative
to the label window. Neither question means the finding is wrong; I just can't tell from
what's published whether the validation design carries the claim as stated, and that's a
one-line fix in the next version of the paper.

In [1]:
import os
import numpy as np
import pandas as pd
import sklearn

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupKFold, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

SEED = 42
np.random.seed(SEED)

if os.getcwd().replace("\\", "/").endswith("work/notebooks"):
    os.chdir("../..")

CSV_PATH = "data/raw/content_refresh_anonymized.csv"
assert os.path.exists(CSV_PATH), "starter CSV not found -- are you at the repo root?"
df = pd.read_csv(CSV_PATH)

# --- Rebuild the Week-5 population, label, and final (permissive) feature set ---
ranked = df["avg_position"] > 0
can_decline = df["impressions_prev_30d"] > 0
pop = df[ranked & can_decline].reset_index(drop=True).copy()
y = pop["trend_direction"].str.lower().eq("down").to_numpy()
groups = pop["client_id"].to_numpy()

LEAK_COLS = ["trend_direction", "trend_pct", "impressions_last_30d", "impressions_prev_30d"]
DROP_ALWAYS = ["content_id", "client_id"] + LEAK_COLS
FEATURES = [c for c in pop.columns if c not in DROP_ALWAYS]
WINDOW_COLS = ["clicks_last_30d", "clicks_prev_30d", "sessions_last_30d", "sessions_prev_30d"]

print(f"population: {len(pop):,} rows | permissive features: {len(FEATURES)} | base rate: {y.mean():.3f}")


def build_pipe(model, cols, frame=None):
    frame = pop if frame is None else frame
    num = [c for c in cols if pd.api.types.is_numeric_dtype(frame[c])]
    cat = [c for c in cols if c not in num]
    pre = ColumnTransformer([
        ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), num),
        ("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=20), cat),
    ])
    return Pipeline([("pre", pre), ("model", model)])


def cv_auc(cols, splitter, use_groups=True, frame=None, target=None, grp=None):
    frame = pop if frame is None else frame
    target = y if target is None else target
    grp = groups if grp is None else grp
    X = frame[cols]
    scores = []
    it = splitter.split(X, target, grp) if use_groups else splitter.split(X, target)
    for tr, te in it:
        pipe = build_pipe(
            RandomForestClassifier(n_estimators=300, min_samples_leaf=5, n_jobs=-1, random_state=SEED),
            cols, frame,
        )
        pipe.fit(X.iloc[tr], target[tr])
        p = pipe.predict_proba(X.iloc[te])[:, 1]
        scores.append(roc_auc_score(target[te], p))
    return scores


GKF = GroupKFold(n_splits=5)
SKF = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
print("setup done | sklearn", sklearn.__version__)

population: 26,604 rows | permissive features: 38 | base rate: 0.611
setup done | sklearn 1.9.0


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

My Week-5 model was already grouped by `client_id`, never split randomly — so "before" here
means re-running the exact same permissive RandomForest under a naive random split instead,
and comparing it to the grouped number I actually shipped.

**Before (naive random split):** mean AUC **0.767** across 5 folds (0.760-0.773).
**After (grouped split by client_id, what I shipped):** mean AUC **0.645** across 5 folds
(0.624-0.684).

**Gap: +0.122.** That's a much bigger gap than my Week-3 leakage audit found on a different
feature set and model (StratifiedKFold vs GroupKFold there was only +0.008, "essentially
nothing"). I don't think that's a contradiction — it's the honest answer changing because
the setup changed. Week 3 used 13 features and Logistic Regression; Week 5 uses 38 features
and a 300-tree Random Forest, which has far more capacity to carve out client-specific
decision boundaries a linear model can't reach. More features and more model flexibility
means more surface area for client memorization, and this gap is the direct measurement of
that: 0.122 AUC points of my model's apparent skill is really "recognizing which client this
row belongs to," not predicting decline. The number I stand behind is still the grouped one,
0.645 — but I was wrong to assume the near-zero Week-3 gap would carry over here just
because it's the same dataset family. It doesn't generalize across model choices, and I
should check this gap again any time I change the feature set or model complexity, not just
once.

In [2]:
auc_grouped = cv_auc(FEATURES, GKF, use_groups=True)
auc_random = cv_auc(FEATURES, SKF, use_groups=False)

print("--- BEFORE: naive random split (rows shuffled across clients) ---")
print(f"per-fold AUC : {[round(s, 3) for s in auc_random]}")
print(f"mean AUC     : {np.mean(auc_random):.3f}")

print("\n--- AFTER: grouped split by client_id (what I actually shipped) ---")
print(f"per-fold AUC : {[round(s, 3) for s in auc_grouped]}")
print(f"mean AUC     : {np.mean(auc_grouped):.3f}")

print(f"\ngap (random - grouped): {np.mean(auc_random) - np.mean(auc_grouped):+.3f}")

--- BEFORE: naive random split (rows shuffled across clients) ---
per-fold AUC : [0.76, 0.767, 0.773, 0.768, 0.767]
mean AUC     : 0.767

--- AFTER: grouped split by client_id (what I actually shipped) ---
per-fold AUC : [0.684, 0.626, 0.624, 0.629, 0.664]
mean AUC     : 0.645

gap (random - grouped): +0.122


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

Running the Week-3 attack checklist against my shipped feature set (permissive, 38 columns,
RandomForest).

**Test A — no label-derived column reached FEATURES.** Pass. `LEAK_COLS` (`trend_direction`,
`trend_pct`, `impressions_last_30d`, `impressions_prev_30d`) are asserted absent. Same caveat
as Week 3: this catches a copy-paste slip, not a subtle leak.

**Test B — put the label's raw material back and watch the score break.** Honest: 0.645.
With `trend_pct` added back: **1.000**. Exactly as expected — `trend_pct` isn't a feature,
it's the label with different arithmetic. This also confirms my test harness itself isn't
broken (per the skill's "how to verify": if adding a known-leaky column doesn't jump the
score, the harness is the thing that's wrong).

**Test C — grouped vs. random split.** Covered in full in Section 2: +0.122 gap. Restated
here because it belongs on this checklist too, not just as a curiosity — a gap this size on
my final feature set means I'd be misreporting the model's real skill by 19% (0.767 vs 0.645
relative) if I'd only checked a random split once and moved on.

**Test D — the degenerate-label population.** In Week 5 I excluded 2,191 rows where
`impressions_prev_30d == 0` (decline is arithmetically impossible) before scoring anything.
Re-including them here: AUC rises from **0.645 to 0.686** (+0.041). Smaller effect than
Week 3's version of this test (+0.325, because 61% of that dataset was in the
impossible-to-decline bucket vs. ~7.6% of mine), but the same mechanism, and the same
conclusion — the exclusion was correct, and the size of the effect depends on how much of
the population the arithmetic trap actually covers, which I should check per-dataset rather
than assume.

**Test E (my addition, not a direct Week-3 mapping) — the window-overlap columns.**
`clicks_last_30d`, `clicks_prev_30d`, `sessions_last_30d`, `sessions_prev_30d` are legal
under Test A/B (they're not the label), but they share a time window with the label's own
`impressions_last_30d`/`impressions_prev_30d` comparison. Stripping them (the "strict" set
from Week 5) drops AUC from 0.645 to **0.611** (-0.034). That's real predictive value being
carried by columns that measure the same window as the outcome — not disqualifying on their
own the way `trend_pct` is, but the honest read is that part of my 0.645 is riding on
same-window correlation rather than a leading indicator a decision-maker would actually have
in hand before the window closes. I'm not changing what I shipped (permissive, 0.645) based
on this alone, but I'm flagging it rather than letting the headline number stand unqualified.

**Where this leaves the final feature set:** 0.645 is real, out-of-fold, grouped, and
survives the label-swap test. It is not fully clean of same-window correlation (Test E), and
19% of what a careless random-split evaluation would have reported (0.767) is client
memorization rather than signal (Test C). Both are disclosed, neither is disqualifying, and
both are why I'm reporting 0.645 with these caveats rather than a bigger number without
them.

In [3]:
print("--- Test A: no LEAK_COLS in FEATURES ---")
leaky_present = [c for c in FEATURES if c in LEAK_COLS]
assert not leaky_present, f"leaky columns present: {leaky_present}"
print("PASS: no label-derived column in FEATURES")

print("\n--- Test B: add back a leak column ---")
auc_honest = np.mean(cv_auc(FEATURES, GKF, use_groups=True))
auc_leaky = np.mean(cv_auc(FEATURES + ["trend_pct"], GKF, use_groups=True))
print(f"honest (permissive)      : {auc_honest:.3f}")
print(f"honest + trend_pct       : {auc_leaky:.3f}")
print(f"jump: {auc_leaky - auc_honest:+.3f}")

print("\n--- Test C: grouped vs random split (see Section 2 for full discussion) ---")
auc_random = np.mean(cv_auc(FEATURES, SKF, use_groups=False))
print(f"grouped : {auc_honest:.3f}")
print(f"random  : {auc_random:.3f}")
print(f"gap     : {auc_random - auc_honest:+.3f}")

print("\n--- Test D: degenerate-label population (impossible-to-decline rows) ---")
ranked_all = df[df["avg_position"] > 0].reset_index(drop=True).copy()
y_incl = ranked_all["trend_direction"].str.lower().eq("down").to_numpy()
groups_incl = ranked_all["client_id"].to_numpy()
impossible_n = int((ranked_all["impressions_prev_30d"] == 0).sum())
print(f"impossible-to-decline rows re-included: {impossible_n:,}")
auc_incl = np.mean(cv_auc(FEATURES, GKF, use_groups=True, frame=ranked_all, target=y_incl, grp=groups_incl))
print(f"AUC including them : {auc_incl:.3f}")
print(f"AUC excluding them  (honest, Week-5 population): {auc_honest:.3f}")
print(f"inflation: {auc_incl - auc_honest:+.3f}")

print("\n--- Test E: window-overlap columns (strict vs permissive) ---")
strict = [c for c in FEATURES if c not in WINDOW_COLS]
auc_strict = np.mean(cv_auc(strict, GKF, use_groups=True))
print(f"permissive (with window cols)   : {auc_honest:.3f}")
print(f"strict (window cols dropped)    : {auc_strict:.3f}")
print(f"value carried by window cols    : {auc_honest - auc_strict:+.3f}")

--- Test A: no LEAK_COLS in FEATURES ---
PASS: no label-derived column in FEATURES

--- Test B: add back a leak column ---


honest (permissive)      : 0.645
honest + trend_pct       : 1.000
jump: +0.355

--- Test C: grouped vs random split (see Section 2 for full discussion) ---


grouped : 0.645
random  : 0.767
gap     : +0.122

--- Test D: degenerate-label population (impossible-to-decline rows) ---
impossible-to-decline rows re-included: 2,191


AUC including them : 0.686
AUC excluding them  (honest, Week-5 population): 0.645
inflation: +0.040

--- Test E: window-overlap columns (strict vs permissive) ---


permissive (with window cols)   : 0.645
strict (window cols dropped)    : 0.611
value carried by window cols    : +0.034


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

**Original (Week 5, Section 4):** "The model isn't just agreeing with the rule faster; it's
adding real signal precisely where the hand-rule goes silent, which is the actual case for
shipping it."

That's the boldest sentence I wrote all internship — it says the model is definitively
better than the rule and settles the shipping decision. This week's own audit gives me a
concrete reason to walk that back rather than just softening the tone for its own sake: the
same model I was defending has a measured 0.122 AUC gap between random and grouped splits
(Section 2/3, Test C), and a measured 0.034 AUC drop when same-window columns are removed
(Test E). Both were true when I wrote the original sentence; I just hadn't measured them
yet.

**Rewrite:** "On the rows where the Week-4 rule scores 0.0 (no opinion), the RandomForest/
permissive model measured an out-of-fold AUC of 0.662 against a 60.1% true-decline rate —
directional evidence that it captures signal the hand-rule misses entirely. That result
should be read as decision-support for further validation, not confirmation that the model
is ready to fully replace the rule: the same model's grouped-vs-random AUC gap is 0.122,
meaning a meaningful share of its overall performance reflects client-specific pattern-
memorization rather than a fully generalizable signal, and roughly 0.034 AUC is carried by
features that share a time window with the label rather than leading it. None of that
reverses the finding — the rule-is-silent result still holds — but "the actual case for
shipping it" overstated what one held-out subset can prove.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.